In [2]:
# importando as bibliotecas
import numpy as np
import matplotlib.pyplot as plt

In [3]:
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, Dense, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras import utils as np_utils
from tensorflow.keras.preprocessing.image import ImageDataGenerator

2026-02-24 18:36:26.726555: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-24 18:36:27.066092: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-24 18:36:32.373547: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [4]:
# carregando os dados e pré-processamento
(X_treinamento, y_treinamento), (X_teste, y_teste) = mnist.load_data()

In [5]:
X_treinamento = X_treinamento.reshape(X_treinamento.shape[0], 28, 28, 1).astype('float32') / 255
X_teste = X_teste.reshape(X_teste.shape[0], 28, 28, 1).astype('float32') / 255

In [6]:
y_treinamento = np_utils.to_categorical(y_treinamento, 10) # one hot encoding
y_teste = np_utils.to_categorical(y_teste, 10)

In [7]:
# configurando o augmentation
gerador_treinamento = ImageDataGenerator(
    rotation_range=10,# rotaciona
    zoom_range=0.1,# aplica zoom
    shear_range=0.1 # distorce levemente
)

gerador_teste = ImageDataGenerator() # sem augmentation no teste

base_treinamento = gerador_treinamento.flow(X_treinamento, y_treinamento, batch_size=128)
base_teste       = gerador_teste.flow(X_teste, y_teste, batch_size=128)

In [8]:
# criação da rede neural
classificador = Sequential()
classificador.add(InputLayer(shape=(28, 28, 1))) # camada de entrada
classificador.add(Conv2D(32, (3, 3), activation='relu', padding='same'))# camada convolucional
classificador.add(MaxPooling2D(pool_size=(2, 2)))# camada de pooling
classificador.add(Flatten())# transforma a matriz em vetor
classificador.add(Dense(units=128, activation='relu'))# camada densa
classificador.add(Dense(units=10,  activation='softmax'))  # camada de saída

2026-02-24 18:36:48.075824: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [9]:
classificador.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
classificador.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       802,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 804,554 (3.07 MB)

 Trainable params: 804,554 (3.07 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
# treinamento com augmentation
classificador.fit(base_treinamento, epochs=5, validation_data=base_teste)

Epoch 1/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 21s 42ms/step - accuracy: 0.9287 - loss: 0.2452 - val_accuracy: 0.9732 - val_loss: 0.0809
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 16s 34ms/step - accuracy: 0.9743 - loss: 0.0848 - val_accuracy: 0.9826 - val_loss: 0.0509
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 16s 34ms/step - accuracy: 0.9811 - loss: 0.0620 - val_accuracy: 0.9839 - val_loss: 0.0441
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 16s 34ms/step - accuracy: 0.9839 - loss: 0.0525 - val_accuracy: 0.9875 - val_loss: 0.0374
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 16s 34ms/step - accuracy: 0.9866 - loss: 0.0431 - val_accuracy: 0.9849 - val_loss: 0.0452


In [11]:
resultado = classificador.evaluate(X_teste, y_teste)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9849 - loss: 0.0452


In [12]:
resultado

[0.045223087072372437, 0.9848999977111816]